##  Import thư viện và cấu hình

In [1]:
import modelscope
from modelscope.pipelines import pipeline

print(modelscope.__version__)


2026-01-30 14:37:25,923 - modelscope - INFO - PyTorch version 2.8.0 Found.
2026-01-30 14:37:25,928 - modelscope - INFO - Loading ast index from C:\Users\PC\.cache\modelscope\ast_indexer
2026-01-30 14:37:25,928 - modelscope - INFO - No valid ast index found from C:\Users\PC\.cache\modelscope\ast_indexer, generating ast index from prebuilt!
2026-01-30 14:37:26,190 - modelscope - INFO - Loading done! Current index file version is 1.10.0, with md5 787eb45ce793c90138f4413fd4db44a8 and a total number of 946 components indexed
e:\anaconda3\envs\ser\lib\site-packages\modelscope\utils\plugins.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
e:\anaconda3\envs\ser\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See http

1.10.0


In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import soundfile as sf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Cấu hình
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Sử dụng device: {DEVICE}")

# Cấu hình đường dẫn
PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_PATH = os.path.join(PROJECT_ROOT, "DATASET_LABELED")
MODEL_SAVE_PATH = os.path.join(os.getcwd(), "emotion2vec_model")

# Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5

print(f" Dataset path: {DATASET_PATH}")
print(f" Model save path: {MODEL_SAVE_PATH}")

2026-01-30 14:37:57,171 - modelscope - INFO - PyTorch version 2.8.0 Found.
2026-01-30 14:37:57,174 - modelscope - INFO - Loading ast index from C:\Users\PC\.cache\modelscope\ast_indexer
2026-01-30 14:37:57,351 - modelscope - INFO - Loading done! Current index file version is 1.10.0, with md5 787eb45ce793c90138f4413fd4db44a8 and a total number of 946 components indexed
e:\anaconda3\envs\ser\lib\site-packages\modelscope\utils\plugins.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
e:\anaconda3\envs\ser\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Sử dụng device: cpu
 Dataset path: e:\KHMT\N4K2\DATN\DATASET_LABELED
 Model save path: e:\KHMT\N4K2\DATN\TrainModel\emotion2vec_model


##  Load và chuẩn bị dữ liệu

In [2]:
# Tạo danh sách các file audio và label tương ứng
def load_dataset_info(dataset_path):
    """Load thông tin về dataset"""
    audio_files = []
    labels = []
    emotion_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    
    print(f" Tìm thấy {len(emotion_folders)} thư mục cảm xúc: {emotion_folders}")
    
    label_to_idx = {emotion: idx for idx, emotion in enumerate(sorted(emotion_folders))}
    idx_to_label = {idx: emotion for emotion, idx in label_to_idx.items()}
    
    for emotion in emotion_folders:
        emotion_path = os.path.join(dataset_path, emotion)
        files = [f for f in os.listdir(emotion_path) if f.endswith(('.wav', '.mp3', '.flac'))]
        
        for file in files:
            audio_files.append(os.path.join(emotion_path, file))
            labels.append(label_to_idx[emotion])
        
        print(f"  {emotion}: {len(files)} files")
    
    return audio_files, labels, label_to_idx, idx_to_label

# Load dataset
audio_files, labels, label_to_idx, idx_to_label = load_dataset_info(DATASET_PATH)
print(f"\n Tổng số file: {len(audio_files)}")
print(f" Số lượng classes: {len(label_to_idx)}")
print(f"  Label mapping: {label_to_idx}")

 Tìm thấy 5 thư mục cảm xúc: ['ANG', 'ANX', 'HAP', 'NEU', 'SAD']
  ANG: 254 files
  ANX: 203 files
  HAP: 197 files
  NEU: 724 files
  SAD: 172 files

 Tổng số file: 1550
 Số lượng classes: 5
  Label mapping: {'ANG': 0, 'ANX': 1, 'HAP': 2, 'NEU': 3, 'SAD': 4}


In [4]:
# Chia dataset thành train/val/test
X_train, X_temp, y_train, y_temp = train_test_split(
    audio_files, labels, test_size=0.3, random_state=42, stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f" Train set: {len(X_train)} samples")
print(f" Validation set: {len(X_val)} samples")
print(f" Test set: {len(X_test)} samples")

 Train set: 1085 samples
 Validation set: 232 samples
 Test set: 233 samples


##  Load Model Emotion2Vec từ ModelScope

In [ ]:
from modelscope.pipelines import pipeline

print("Đang tải Emotion2Vec model từ ModelScope...")

inference_pipeline = pipeline(
    task="speech_emotion_recognition",
    model="iic/emotion2vec_base_finetuned",
    model_revision="v2.0.4"
)

print("Model đã được tải thành công!")


Đang tải Emotion2Vec model từ ModelScope...


2026-01-30 14:39:18,627 - modelscope - INFO - Use user-specified model revision: v2.0.4
Downloading: 100%|██████████| 3.06k/3.06k [00:00<00:00, 912kB/s]
Downloading: 100%|██████████| 354/354 [00:00<00:00, 195kB/s]
Downloading:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

## Tạo Custom Dataset và DataLoader

In [ ]:
class EmotionDataset(Dataset):
    """Custom Dataset cho audio emotion recognition"""
    
    def __init__(self, audio_files, labels, inference_pipeline):
        self.audio_files = audio_files
        self.labels = labels
        self.inference_pipeline = inference_pipeline
        
    def __len__(self):
        return len(self.audio_files)
    
    def __getitem__(self, idx):
        audio_path = self.audio_files[idx]
        label = self.labels[idx]
        
        try:
            # Extract features bằng Emotion2Vec
            result = self.inference_pipeline(audio_path, output_type="features")
            features = result['feats']  # Features từ Emotion2Vec
            
            # Convert to tensor
            if not isinstance(features, torch.Tensor):
                features = torch.tensor(features, dtype=torch.float32)
            
            # Global average pooling nếu features có nhiều frames
            if len(features.shape) > 1:
                features = features.mean(dim=0)  # [feature_dim]
            
            return features, label
        
        except Exception as e:
            print(f" Lỗi khi xử lý file {audio_path}: {e}")
            # Return zero tensor nếu có lỗi
            return torch.zeros(768), label  # 768 là feature dimension của emotion2vec_base

# Tạo datasets
print(" Đang tạo datasets...")
train_dataset = EmotionDataset(X_train, y_train, inference_pipeline)
val_dataset = EmotionDataset(X_val, y_val, inference_pipeline)
test_dataset = EmotionDataset(X_test, y_test, inference_pipeline)

# Tạo dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Datasets và DataLoaders đã được tạo!")

## Định nghĩa Classification Head

In [ ]:
class EmotionClassifier(nn.Module):
    """
    Classification head trên features của Emotion2Vec
    """
    def __init__(self, input_dim=768, hidden_dim=256, num_classes=5, dropout=0.3):
        super(EmotionClassifier, self).__init__()
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(hidden_dim),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(hidden_dim // 2),
            
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

# Khởi tạo model
num_classes = len(label_to_idx)
model = EmotionClassifier(input_dim=768, num_classes=num_classes).to(DEVICE)

# Loss function và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

print(f" Model được khởi tạo với {num_classes} classes")
print(f" Số lượng parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

##  Training Loop

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train một epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for features, labels in pbar:
        features, labels = features.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100 * correct / total:.2f}%'})
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for features, labels in tqdm(dataloader, desc="Validating", leave=False):
            features, labels = features.to(device), labels.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

print(" Training functions đã được định nghĩa!")

In [ ]:
# Training loop với early stopping
print(" Bắt đầu training...\n")

best_val_acc = 0.0
best_model_path = os.path.join(MODEL_SAVE_PATH, 'best_model.pth')
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

early_stopping_counter = 0

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"{'='*60}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)
    
    # Learning rate scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"\n Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'label_to_idx': label_to_idx,
            'idx_to_label': idx_to_label
        }, best_model_path)
        print(f"    Đã lưu best model! (Val Acc: {val_acc:.2f}%)")
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1
        print(f"    Early stopping counter: {early_stopping_counter}/{EARLY_STOPPING_PATIENCE}")
    
    # Early stopping
    if early_stopping_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\n Early stopping triggered after {epoch+1} epochs")
        break

print(f"\n{'='*60}")
print(f" Training hoàn tất!")
print(f" Best validation accuracy: {best_val_acc:.2f}%")
print(f"{'='*60}")

##  Vẽ Learning Curves

In [ ]:
# Vẽ learning curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_PATH, 'learning_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print(" Learning curves đã được vẽ và lưu!")

##  Đánh giá trên Test Set

In [ ]:
# Load best model
print(" Đang load best model...")
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f" Đã load model từ epoch {checkpoint['epoch']} với Val Acc: {checkpoint['val_acc']:.2f}%")

# Evaluate trên test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in tqdm(test_loader, desc="Testing"):
        features, labels = features.to(DEVICE), labels.to(DEVICE)
        outputs = model(features)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
test_acc = 100 * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
print(f"\n{'='*60}")
print(f" TEST SET ACCURACY: {test_acc:.2f}%")
print(f"{'='*60}\n")

# Classification report
print(" CLASSIFICATION REPORT:\n")
class_names = [idx_to_label[i] for i in range(len(idx_to_label))]
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

In [ ]:
# Vẽ Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_PATH, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

print(" Confusion matrix đã được vẽ và lưu!")

## Lưu Model và Thông Tin

In [ ]:
# Lưu thông tin training
import json

training_info = {
    'test_accuracy': float(test_acc),
    'best_val_accuracy': float(best_val_acc),
    'num_classes': num_classes,
    'label_to_idx': label_to_idx,
    'idx_to_label': {int(k): v for k, v in idx_to_label.items()},
    'num_train_samples': len(X_train),
    'num_val_samples': len(X_val),
    'num_test_samples': len(X_test),
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE
    }
}

info_path = os.path.join(MODEL_SAVE_PATH, 'training_info.json')
with open(info_path, 'w', encoding='utf-8') as f:
    json.dump(training_info, f, indent=4, ensure_ascii=False)

print(f" Thông tin training đã được lưu tại: {info_path}")

# Lưu history
history_path = os.path.join(MODEL_SAVE_PATH, 'training_history.npy')
np.save(history_path, history)
print(f" Training history đã được lưu tại: {history_path}")

print(f"\n{'='*60}")
print(f" HOÀN TẤT!")
print(f"{'='*60}")
print(f" Model và kết quả được lưu tại: {MODEL_SAVE_PATH}")
print(f" Test Accuracy: {test_acc:.2f}%")
print(f"{'='*60}")

In [ ]:
def predict_emotion(audio_path, model, inference_pipeline, idx_to_label, device):
    """
    Dự đoán cảm xúc từ file audio
    """
    model.eval()
    
    try:
        # Extract features bằng Emotion2Vec
        result = inference_pipeline(audio_path, output_type="features")
        features = result['feats']
        
        # Convert to tensor
        if not isinstance(features, torch.Tensor):
            features = torch.tensor(features, dtype=torch.float32)
        
        # Global average pooling
        if len(features.shape) > 1:
            features = features.mean(dim=0)
        
        # Add batch dimension
        features = features.unsqueeze(0).to(device)
        
        # Predict
        with torch.no_grad():
            outputs = model(features)
            probabilities = torch.softmax(outputs, dim=1)
            predicted_class = torch.argmax(probabilities, dim=1).item()
            confidence = probabilities[0][predicted_class].item()
        
        emotion = idx_to_label[predicted_class]
        
        return emotion, confidence, probabilities[0].cpu().numpy()
    
    except Exception as e:
        print(f" Lỗi: {e}")
        return None, None, None


# Demo với một file từ test set
if len(X_test) > 0:
    demo_audio_path = X_test[0]
    true_label = idx_to_label[y_test[0]]
    
    print(f"🎵 Demo file: {os.path.basename(demo_audio_path)}")
    print(f"  True label: {true_label}")
    print(f"\n{'='*50}")
    
    emotion, confidence, probs = predict_emotion(demo_audio_path, model, inference_pipeline, idx_to_label, DEVICE)
    
    if emotion:
        print(f" Predicted: {emotion}")
        print(f" Confidence: {confidence*100:.2f}%")
        print(f"\n{'='*50}")
        print("Xác suất cho tất cả các lớp:")
        for i, prob in enumerate(probs):
            print(f"  {idx_to_label[i]:<10}: {prob*100:5.2f}%")
    
    print(f"{'='*50}")